# 🧪 Emoji Overlay Diagnostic — Lightweight

Standalone test. No audio, Whisper, song map, or Stage 2 code. Existing emoji assets are read-only.

This version tests only a small representative set instead of opening thousands of assets.

In [ ]:
from google.colab import drive
import os,re,json
from pathlib import Path
from PIL import Image,ImageDraw
drive.mount('/content/drive',force_remount=False)
FOLDER='/content/drive/MyDrive/AIgenerated'
EMOJI_ROOT=os.path.join(FOLDER,'assets','emoji')
OUT_ROOT=os.path.join(FOLDER,'emoji_test')
OUT_PATH=os.path.join(OUT_ROOT,'emoji_overlay_test.png')
os.makedirs(OUT_ROOT,exist_ok=True)
print('Emoji root:',EMOJI_ROOT)


In [ ]:
exts={'.png','.jpg','.jpeg','.webp'}
imgs=[p for p in Path(FOLDER).iterdir() if p.is_file() and p.suffix.lower() in exts and not p.name.startswith('_')]
if not imgs: raise RuntimeError('No image found directly in AIgenerated/.')
BASE_IMAGE=str(sorted(imgs,key=lambda p:p.name.lower())[0])
base=Image.open(BASE_IMAGE).convert('RGBA')
base.thumbnail((1080,1920),Image.Resampling.LANCZOS)
print('Canvas:',BASE_IMAGE,base.size)


In [ ]:
TEST_EMOJIS=['💖','🤩','👍','🔥','💧','😀','❤️','😂','🥰','😍','✨','😭','🇮🇳','🇺🇸','👩‍💻','👍🏽','❤️‍🔥']
files=[]
for root,dirs,names in os.walk(EMOJI_ROOT):
    for n in names:
        if Path(n).suffix.lower() in {'.png','.webp','.jpg','.jpeg'}: files.append(os.path.join(root,n))
print('Asset filenames indexed:',len(files))
def norm(s): return s.casefold().replace(' ','').replace('-','').replace('_','')
def candidates_for(emoji):
    cp='-'.join(f'{ord(c):X}' for c in emoji)
    compact=cp.replace('-','')
    keys=[norm(emoji),norm(cp),norm(compact),norm('U+'+cp)]
    return [p for p in files if any(k and (k in norm(Path(p).stem) or norm(Path(p).stem) in k) for k in keys)][:3]
mapping={}
for e in TEST_EMOJIS:
    mapping[e]=candidates_for(e)
    print(repr(e),[Path(x).name for x in mapping[e]])


In [ ]:
W,H=base.size
canvas=base.copy()
draw=ImageDraw.Draw(canvas)
panel_w=min(1000,W-40); panel_h=min(1700,H-80)
panel=Image.new('RGBA',(panel_w,panel_h),(0,0,0,150))
canvas.alpha_composite(panel,((W-panel_w)//2,(H-panel_h)//2))
x0=(W-panel_w)//2+40; y0=(H-panel_h)//2+40; cell_w=max(1,(panel_w-80)//4); cell_h=390
results=[]
for i,e in enumerate(TEST_EMOJIS):
    hits=mapping[e]; row={'emoji':e,'codepoints':[f'U+{ord(c):04X}' for c in e],'matches':hits,'status':'not_found'}
    if hits:
        try:
            em=Image.open(hits[0]).convert('RGBA'); em.thumbnail((220,220),Image.Resampling.LANCZOS)
            col=i%4; rown=i//4; x=x0+col*cell_w+(cell_w-em.width)//2; y=y0+rown*cell_h
            canvas.alpha_composite(em,(x,y)); draw.text((x0+col*cell_w,y+235),f'{e}  '+Path(hits[0]).name[:24],fill='white')
            row['status']='rendered'; row['render_size']=[em.width,em.height]
        except Exception as ex: row['status']='load_error'; row['error']=str(ex)
    else:
        col=i%4; rown=i//4; draw.text((x0+col*cell_w,y0+rown*cell_h+100),f'{e}  NOT FOUND',fill='white')
    results.append(row)
canvas.convert('RGB').save(OUT_PATH,quality=92)
print('Rendered:',sum(r['status']=='rendered' for r in results),'/',len(results))
print('Output:',OUT_PATH)


In [ ]:
report={'canvas':BASE_IMAGE,'emoji_root':EMOJI_ROOT,'tested':TEST_EMOJIS,'results':results}
report_path=os.path.join(OUT_ROOT,'emoji_overlay_test_report.json')
with open(report_path,'w',encoding='utf-8') as f: json.dump(report,f,ensure_ascii=False,indent=2)
print('Report:',report_path)
print('✅ Lightweight emoji test complete.')
